In [11]:
from experiments.dj.result_tables import FlowPriorResult
from experiments.dj.prior_tables import FlowPriorConfig
from experiments.dj.dataloader_tables import DataLoaderConfig
from experiments.dj.trainer_tables import FPTrainerConfig
from experiments.dj.evaluation_tables import FlowPriorEval
from task_transfer.evaluation.evaluate_generative_model import evaluate_flow_prior, compute_logl
from task_transfer.ml_lib.data_loading import build_dataloaders

import gensn.distributions as G

import torch
import torch.distributions as D

In [2]:
fp_cols = [col for col in FlowPriorConfig.heading if col != "id"]
dl_cols = [col for col in DataLoaderConfig.heading if col != "id"]
trainer_cols = [col for col in FPTrainerConfig.heading if col != "id"]

result_table = (
    FlowPriorResult()
    * FlowPriorConfig.proj(fp_id="id", *fp_cols)
    * DataLoaderConfig.proj(dl_id="id", *dl_cols)
    * FPTrainerConfig.proj(trainer_id="id", *trainer_cols)
)

In [3]:
best_val = (
    result_table
    & "data_fname = '/src/project/data/synthetic/haefner_2afc/flat_haefner_dataset.pkl'"
).fetch(download_path="/tmp", order_by="val_ll_mean DESC", limit=1, as_dict=True)[0]

In [4]:
flow_path = best_val["model"]
flow_model = torch.load(flow_path)
flow_model.eval()
data_path = best_val["data_fname"]
train_prop = best_val["train_prop"]
val_prop = best_val["val_prop"]
batch_size = int(best_val["batch_size"])
train_loader, val_loader, test_loader = build_dataloaders(
    data_path, train_prop, val_prop, batch_size
)

In [7]:
response_sample, image_sample = next(iter(train_loader))
response_dim = response_sample.shape[-1]

In [12]:
true_prior = G.IndependentExponential(
    rate=torch.ones(response_dim)
)

In [14]:
data_dim = 0
device = torch.device("cuda:1")
reduction = "none"
uncertainty = "sem"
normalize = "none"
unit = "nats"

In [20]:
true_train_lp, true_train_sem = compute_logl(
    model=true_prior,
    data_loader=train_loader,
    data_dim=data_dim,
    device=device,
    reduction=reduction,
    uncertainty=uncertainty,
    normalize=normalize,
    unit=unit,
)
true_val_lp, true_val_sem = compute_logl(
    model=true_prior,
    data_loader=val_loader,
    data_dim=data_dim,
    device=device,
    reduction=reduction,
    uncertainty=uncertainty,
    normalize=normalize,
    unit=unit,
)
true_test_lp, true_test_sem = compute_logl(
    model=true_prior,
    data_loader=test_loader,
    data_dim=data_dim,
    device=device,
    reduction=reduction,
    uncertainty=uncertainty,
    normalize=normalize,
    unit=unit,
)

In [17]:
true_train_lp.mean(), true_val_lp.mean(), true_test_lp.mean()

(tensor(-44.9860, device='cuda:1'),
 tensor(-45.1202, device='cuda:1'),
 tensor(-44.9191, device='cuda:1'))

In [21]:
true_train_sem, true_val_sem, true_test_sem

(0.9035487174987793, 1.6894515752792358, 2.3812215328216553)

In [19]:
best_val["train_ll_mean"], best_val["val_ll_mean"], best_val["test_ll_mean"]

(-45.06998062133789, -45.21186447143555, -45.21186447143555)

In [22]:
best_val["train_ll_sem"], best_val["val_ll_sem"], best_val["test_ll_sem"]

(0.9043867588043213, 1.6942139863967896, 1.6942139863967896)

In [ ]:
s